#### Memuat Dataset Hasil Data Preparation

Tahap ini bertujuan untuk memuat dataset master hasil proses data preparation yang akan digunakan pada tahap pemodelan machine learning. Dataset dimuat dalam format Parquet atau CSV sebagai alternatif, kemudian dilakukan penyesuaian tipe data tanggal dan pengurutan berdasarkan lokasi serta waktu untuk menjaga konsistensi struktur data panel spasial-temporal.

In [ ]:
# Import library dan konfigurasi output

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


# Tentukan folder dataset dan lokasi file input

candidate_dirs = [
    Path("../data/processed"),
    Path("ml/data/processed"),
    Path("data/processed"),
]

DIR_PROCESSED = next(
    (path for path in candidate_dirs if path.exists()),
    candidate_dirs[0],
)

parquet_path = DIR_PROCESSED / "master_dataset.parquet"
csv_path = DIR_PROCESSED / "master_dataset.csv"


# Muat dataset hasil data preparation

if parquet_path.exists():
    # Gunakan Parquet karena lebih efisien untuk pemrosesan data besar
    df = pd.read_parquet(parquet_path)
    data_source = parquet_path

else:
    # Gunakan CSV sebagai fallback apabila Parquet tidak tersedia
    df = pd.read_csv(
        csv_path,
        parse_dates=["date"],
    )
    data_source = csv_path

df["date"] = pd.to_datetime(df["date"])


# Urutkan data untuk menjaga struktur panel berdasarkan lokasi dan waktu

df = (
    df.sort_values(
        ["location_id", "date"]
    )
    .reset_index(drop=True)
)


# Tentukan variabel target model

TARGET = "GHI"


# Tampilkan ringkasan dan sampel dataset

print(f"Sumber data   : {data_source}")
print(f"Jumlah baris  : {len(df):,}")
print(f"Jumlah kolom  : {df.shape[1]}")
print(f"Jumlah lokasi : {df['location_id'].nunique()}")
print(
    f"Rentang data  : "
    f"{df['date'].min().date()} "
    f"sampai "
    f"{df['date'].max().date()}"
)
print(f"Target        : {TARGET}")

df.head()

Sumber data   : ..\data\processed\master_dataset.parquet
Jumlah baris  : 73,040
Jumlah kolom  : 11
Jumlah lokasi : 40
Rentang data  : 2021-01-01 sampai 2025-12-31
Target        : GHI


,location_id,lat,lon,date,GHI,DHI,DNI,TEMP,RH,PRECIP,meteo_extrapolated
0,-5.5000_105.5000,-5.5,105.5,2021-01-01,5.9597,2.6189,3.7409,27.00,82.81,5.64,True
1,-5.5000_105.5000,-5.5,105.5,2021-01-02,5.7943,3.1807,2.7691,26.67,84.42,10.10,True
2,-5.5000_105.5000,-5.5,105.5,2021-01-03,4.3733,2.9808,0.7447,27.01,82.85,13.56,True
3,-5.5000_105.5000,-5.5,105.5,2021-01-04,5.3333,3.0636,2.5296,26.97,82.29,10.85,True
4,-5.5000_105.5000,-5.5,105.5,2021-01-05,3.0929,2.2714,0.1898,26.73,87.08,3.04,True


#### Rekayasa Fitur Temporal

Tahap ini bertujuan untuk membentuk fitur temporal yang dapat membantu model mempelajari pola musiman radiasi surya. Informasi tanggal diekstraksi menjadi beberapa komponen waktu, kemudian dilakukan transformasi siklikal menggunakan fungsi sinus dan cosinus agar sifat periodik pada bulan dan hari dalam satu tahun dapat direpresentasikan dengan baik.

Fitur temporal yang dihasilkan akan digunakan sebagai masukan model machine learning pada seluruh skenario eksperimen.

In [ ]:
# Siapkan dataset untuk proses feature engineering

fe = df.copy()


# Ekstraksi fitur waktu dan encoding siklikal untuk pola musiman

fe["year"] = fe["date"].dt.year
fe["month"] = fe["date"].dt.month
fe["day_of_year"] = fe["date"].dt.dayofyear

fe["month_sin"] = np.sin(
    2 * np.pi * fe["month"] / 12.0
)

fe["month_cos"] = np.cos(
    2 * np.pi * fe["month"] / 12.0
)

fe["doy_sin"] = np.sin(
    2 * np.pi * fe["day_of_year"] / 365.25
)

fe["doy_cos"] = np.cos(
    2 * np.pi * fe["day_of_year"] / 365.25
)


# Kelompokkan fitur temporal

TEMPORAL_FEATURES = [
    "month",
    "day_of_year",
    "month_sin",
    "month_cos",
    "doy_sin",
    "doy_cos",
]


# Tampilkan fitur temporal dan contoh hasil feature engineering

print("Temporal features:")
print(TEMPORAL_FEATURES)

fe[
    [
        "date",
        *TEMPORAL_FEATURES,
    ]
].head()

Temporal features:
['month', 'day_of_year', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos']


,date,month,day_of_year,month_sin,month_cos,doy_sin,doy_cos
0,2021-01-01,1,1,0.5,0.866025,0.017202,0.999852
1,2021-01-02,1,2,0.5,0.866025,0.034398,0.999408
2,2021-01-03,1,3,0.5,0.866025,0.051584,0.998669
3,2021-01-04,1,4,0.5,0.866025,0.068755,0.997634
4,2021-01-05,1,5,0.5,0.866025,0.085906,0.996303


#### Definisi Kelompok Fitur Model

Tahap ini bertujuan untuk mengelompokkan variabel masukan berdasarkan karakteristik informasinya sebelum digunakan dalam eksperimen pemodelan. Fitur dibagi menjadi beberapa kelompok, yaitu fitur spasial, temporal, meteorologis, dan irradiance untuk mendukung perbandingan antar skenario penggunaan fitur.

In [ ]:
# Kelompokkan fitur berdasarkan jenis data

SPATIAL_FEATURES = [
    "lat",
    "lon",
]

METEO_FEATURES = [
    "TEMP",
    "RH",
    "PRECIP",
]

IRRADIANCE_FEATURES = [
    "DHI",
    "DNI",
]


# Gabungkan fitur spasial, temporal, dan meteorologis sebagai fitur dasar

BASE_FEATURES = (
    SPATIAL_FEATURES
    + TEMPORAL_FEATURES
    + METEO_FEATURES
)


# Tampilkan komposisi dan jumlah fitur

print("Spatial features:")
print(SPATIAL_FEATURES)

print("\nTemporal features:")
print(TEMPORAL_FEATURES)

print("\nMeteorological features:")
print(METEO_FEATURES)

print("\nIrradiance features:")
print(IRRADIANCE_FEATURES)

print(f"\nTotal base features: {len(BASE_FEATURES)}")

Spatial features:
['lat', 'lon']

Temporal features:
['month', 'day_of_year', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos']

Meteorological features:
['TEMP', 'RH', 'PRECIP']

Irradiance features:
['DHI', 'DNI']

Total base features: 11


#### Penanganan Missing Value

Tahap ini bertujuan untuk memastikan kualitas data sebelum digunakan pada proses pelatihan model.

Baris yang memiliki nilai kosong pada variabel target (GHI) dihapus karena model supervised learning memerlukan target yang lengkap selama proses pelatihan. Sementara itu, missing value pada variabel fitur tidak diimputasi karena algoritma LightGBM dan XGBoost mampu menangani nilai yang hilang (*missing value*) secara native selama proses pembentukan pohon keputusan.

In [ ]:
# Tangani missing value pada target model

n_rows_before = len(fe)

n_target_missing = int(
    fe[TARGET]
    .isna()
    .sum()
)

fe = (
    fe.dropna(
        subset=[TARGET]
    )
    .reset_index(drop=True)
)


# Tampilkan perubahan jumlah data setelah pembersihan target

print(
    f"Missing value pada target ({TARGET}) : "
    f"{n_target_missing}"
)

print(
    f"Jumlah data : "
    f"{n_rows_before:,} -> {len(fe):,} baris"
)


# Periksa missing value pada seluruh fitur model

feature_missing = (
    fe[
        BASE_FEATURES
        + IRRADIANCE_FEATURES
    ]
    .isna()
    .sum()
)

feature_missing = (
    feature_missing[
        feature_missing > 0
    ]
    .sort_values(
        ascending=False
    )
)


# Tampilkan fitur yang masih memiliki missing value

print("\n=== Missing Value pada Fitur ===")

if feature_missing.empty:
    print("Tidak terdapat missing value pada seluruh fitur.")
else:
    display(
        feature_missing.to_frame(
            name="n_missing"
        )
    )

Missing value pada target (GHI) : 0
Jumlah data : 73,040 -> 73,040 baris

=== Missing Value pada Fitur ===


,n_missing
DHI,40
DNI,40


#### Definisi Skenario Fitur Model

Tahap ini bertujuan untuk mendefinisikan beberapa skenario eksperimen berdasarkan kombinasi fitur yang digunakan oleh model machine learning.

Dua skenario yang dibandingkan adalah:
- **METEO**: menggunakan informasi spasial, temporal, dan meteorologis tanpa variabel irradiance turunan.
- **FULL**: menggunakan seluruh fitur METEO ditambah komponen irradiance (DHI dan DNI).

Pemisahan skenario ini dilakukan untuk mengevaluasi seberapa besar kontribusi variabel irradiance tambahan terhadap kemampuan model dalam memprediksi Global Horizontal Irradiance (GHI).

In [ ]:
# Definisikan komposisi fitur untuk setiap skenario eksperimen

FEATURE_SETS = {
    "METEO": BASE_FEATURES,
    "FULL": BASE_FEATURES + IRRADIANCE_FEATURES,
}


# Tampilkan jumlah dan daftar fitur setiap skenario

for scenario_name, feature_list in FEATURE_SETS.items():

    print(f"=== Scenario: {scenario_name} ===")
    print(f"Number of features : {len(feature_list)}")

    display(
        pd.DataFrame(
            {
                "Feature": feature_list,
            }
        )
    )

=== Scenario: METEO ===
Number of features : 11


,Feature
0,lat
1,lon
2,month
3,day_of_year
4,month_sin
5,month_cos
6,doy_sin
7,doy_cos
8,TEMP
9,RH


=== Scenario: FULL ===
Number of features : 13


,Feature
0,lat
1,lon
2,month
3,day_of_year
4,month_sin
5,month_cos
6,doy_sin
7,doy_cos
8,TEMP
9,RH


#### Pembagian Data Training dan Testing Berbasis Waktu

Tahap ini bertujuan untuk membagi dataset menjadi data pelatihan (**training set**) dan data pengujian (**testing set**) menggunakan pendekatan time-based split.

Data tahun 2021–2024 digunakan sebagai training set untuk membangun model, sedangkan data tahun 2025 digunakan sebagai hold-out test set untuk mengevaluasi kemampuan generalisasi model terhadap periode waktu yang belum pernah dilihat sebelumnya.

Pendekatan ini dipilih karena data radiasi surya merupakan data deret waktu, sehingga pembagian acak (random split) berpotensi menyebabkan kebocoran informasi temporal (*data leakage*).

In [ ]:
# Tentukan periode training dan testing

TRAIN_YEARS = [
    2021,
    2022,
    2023,
    2024,
]

TEST_YEARS = [
    2025,
]


# Validasi tahun dan tandai data berdasarkan split

VALID_YEARS = TRAIN_YEARS + TEST_YEARS

assert (
    set(fe["year"].unique())
    .issubset(VALID_YEARS)
), "Dataset mengandung tahun di luar konfigurasi split."

fe["split"] = np.where(
    fe["year"].isin(TEST_YEARS),
    "test",
    "train",
)


# Ringkas hasil pembagian dataset

split_summary = (
    fe.groupby("split")
    .agg(
        n_rows=("date", "size"),
        start_date=("date", "min"),
        end_date=("date", "max"),
        n_locations=("location_id", "nunique"),
    )
    .reindex(["train", "test"])
)

display(split_summary)

print(
    f"Training data : "
    f"{(fe['split'] == 'train').sum():,} baris"
)

print(
    f"Testing data  : "
    f"{(fe['split'] == 'test').sum():,} baris"
)

print(
    f"Test proportion: "
    f"{(fe['split'] == 'test').mean() * 100:.1f}%"
)

,n_rows,start_date,end_date,n_locations
split,,,,
train,58440,2021-01-01,2024-12-31,40
test,14600,2025-01-01,2025-12-31,40


Training data : 58,440 baris
Testing data  : 14,600 baris
Test proportion: 20.0%


## 7. Validasi: Anti-Leakage & Sanity Check

Memastikan target tidak menjadi fitur, tidak ada irisan tanggal train–test, dan fitur temporal bebas NaN/inf serta berada di rentang valid.

In [ ]:
# Siapkan seluruh fitur dan tanggal untuk validasi feature engineering

all_features = sorted(
    set(FEATURE_SETS["FULL"])
)

train_dates = set(
    fe.loc[
        fe["split"] == "train",
        "date",
    ]
)

test_dates = set(
    fe.loc[
        fe["split"] == "test",
        "date",
    ]
)

temporal_is_finite = np.isfinite(
    fe[TEMPORAL_FEATURES].to_numpy()
).all()


# Validasi struktur fitur, temporal split, dan encoding

validation_checks = {

    "Target tidak digunakan sebagai fitur":
        TARGET not in all_features,

    "Tidak ada overlap tanggal train-test":
        len(train_dates & test_dates) == 0,

    "Urutan waktu train sebelum test":
        max(train_dates) < min(test_dates),

    "Fitur temporal bebas NaN dan infinity":
        bool(temporal_is_finite),

    "Seluruh fitur tersedia":
        all(feature in fe.columns for feature in all_features),

    "Encoding siklikal berada pada rentang [-1, 1]":
        (
            fe[
                [
                    "month_sin",
                    "month_cos",
                    "doy_sin",
                    "doy_cos",
                ]
            ]
            .abs()
            .le(1.0 + 1e-9)
            .all()
            .all()
        ),
}


# Tampilkan dan validasi seluruh pemeriksaan

print("Hasil validasi feature engineering:")

for check_name, passed in validation_checks.items():
    status = "OK" if passed else "GAGAL"
    print(f"[{status}] {check_name}")

assert all(validation_checks.values()), (
    "Terdapat validasi feature engineering yang gagal."
)

print("\nSemua validasi feature engineering berhasil dilewati.")


# Pastikan seluruh lokasi memiliki data train dan test

location_split = (
    fe.groupby("location_id")["split"]
    .nunique()
)

assert (location_split == 2).all(), (
    "Terdapat lokasi yang tidak memiliki data train atau test."
)

print("Semua lokasi memiliki data pada train dan test.")

Hasil validasi feature engineering:
[OK] Target tidak digunakan sebagai fitur
[OK] Tidak ada overlap tanggal train-test
[OK] Urutan waktu train sebelum test
[OK] Fitur temporal bebas NaN dan infinity
[OK] Seluruh fitur tersedia
[OK] Encoding siklikal berada pada rentang [-1, 1]

Semua validasi feature engineering berhasil dilewati.
Semua lokasi memiliki data pada train dan test.


#### Simpan Tabel Fitur & Konfigurasi

Tahap ini bertujuan untuk menyimpan dataset hasil feature engineering beserta konfigurasi eksperimen yang akan digunakan pada proses pelatihan dan evaluasi model. Dataset disimpan dalam format CSV dan Parquet, sedangkan informasi mengenai target, kelompok fitur, skenario eksperimen, serta pembagian data disimpan dalam berkas JSON agar seluruh notebook berikutnya menggunakan konfigurasi yang konsisten dan terdokumentasi.

In [ ]:
# Susun dataset akhir dan urutan kolom hasil feature engineering

OUTPUT_COLUMNS = (
    [
        "location_id",
        "date",
        "year",
        "split",
    ]
    + [TARGET]
    + sorted(set(FEATURE_SETS["FULL"]))
    + (
        ["meteo_extrapolated"]
        if "meteo_extrapolated" in fe.columns
        else []
    )
)

OUTPUT_COLUMNS = list(
    dict.fromkeys(OUTPUT_COLUMNS)
)

feature_dataset = (
    fe[OUTPUT_COLUMNS]
    .copy()
)


# Simpan dataset hasil feature engineering

feature_csv_path = (
    DIR_PROCESSED /
    "features_dataset.csv"
)

feature_parquet_path = (
    DIR_PROCESSED /
    "features_dataset.parquet"
)

feature_dataset.to_csv(
    feature_csv_path,
    index=False,
)

print(
    f"[OK] CSV      -> {feature_csv_path} "
    f"({feature_csv_path.stat().st_size / 1e6:.2f} MB)"
)

try:

    feature_dataset.to_parquet(
        feature_parquet_path,
        index=False,
    )

    print(
        f"[OK] Parquet -> {feature_parquet_path} "
        f"({feature_parquet_path.stat().st_size / 1e6:.2f} MB)"
    )

except Exception as error:

    print(
        f"[!] Parquet dilewati "
        f"({type(error).__name__})."
    )

    print(
        "Install dengan: pip install pyarrow"
    )


# Susun dan simpan konfigurasi eksperimen

feature_config = {

    "target": TARGET,

    "feature_sets": FEATURE_SETS,

    "base_features": BASE_FEATURES,
    "irradiance_features": IRRADIANCE_FEATURES,
    "spatial_features": SPATIAL_FEATURES,
    "temporal_features": TEMPORAL_FEATURES,
    "meteo_features": METEO_FEATURES,

    "split": {
        "train_years": TRAIN_YEARS,
        "test_years": TEST_YEARS,
        "column": "split",
    },

    "dataset_file": feature_parquet_path.name,
    "n_rows": int(len(feature_dataset)),
    "n_locations": int(
        feature_dataset["location_id"].nunique()
    ),
}


config_path = (
    DIR_PROCESSED /
    "feature_config.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        feature_config,
        file,
        indent=4,
    )


# Tampilkan hasil penyimpanan dan konfigurasi eksperimen

print(f"[OK] Config   -> {config_path}")

print("\nIsi feature_config.json")
print(
    json.dumps(
        feature_config,
        indent=4,
    )
)

[OK] CSV      -> ..\data\processed\features_dataset.csv (13.38 MB)
[OK] Parquet -> ..\data\processed\features_dataset.parquet (1.55 MB)
[OK] Config   -> ..\data\processed\feature_config.json

Isi feature_config.json
{
    "target": "GHI",
    "feature_sets": {
        "METEO": [
            "lat",
            "lon",
            "month",
            "day_of_year",
            "month_sin",
            "month_cos",
            "doy_sin",
            "doy_cos",
            "TEMP",
            "RH",
            "PRECIP"
        ],
        "FULL": [
            "lat",
            "lon",
            "month",
            "day_of_year",
            "month_sin",
            "month_cos",
            "doy_sin",
            "doy_cos",
            "TEMP",
            "RH",
            "PRECIP",
            "DHI",
            "DNI"
        ]
    },
    "base_features": [
        "lat",
        "lon",
        "month",
        "day_of_year",
        "month_sin",
        "month_cos",
        "doy_sin",
